
# Business Scenario: EcoDelivery Logistics

## Overview
EcoDelivery, a startup specializing in eco-friendly deliveries, wants to optimize its electric vehicle (EV) delivery network to reduce operational costs while meeting customer demands. 
The company operates three hubs and delivers packages to distribution centers in various cities.

---

## Hubs
EcoDelivery operates three hubs:
- **San Francisco** (SFO)
- **Denver** (DEN)
- **Austin** (AUS)

Each hub has a limited number of delivery vans and can transport a fixed number of packages:

| **Hub**          | **Capacity (Packages)** |
|-------------------|-------------------------|
| San Francisco     | 300                    |
| Denver            | 250                    |
| Austin            | 200                    |

---

## Distribution Centers (Demand Points)
Packages need to be delivered to the following distribution centers:

| **Distribution Center** | **Demand (Packages)** |
|--------------------------|-----------------------|
| Seattle                 | 200                   |
| Chicago                 | 150                   |
| Houston                 | 250                   |

---

## Transportation Costs
The cost per package to transport from a hub to a distribution center depends on the distance and EV charging costs. The following table summarizes the transportation costs:

| **Hub**       | **Distribution Center** | **Cost per Package ($)** |
|---------------|--------------------------|--------------------------|
| San Francisco | Seattle                 | 80                       |
| San Francisco | Chicago                 | 100                      |
| Denver        | Chicago                 | 70                       |
| Denver        | Houston                 | 90                       |
| Austin        | Houston                 | 50                       |
| Austin        | Seattle                 | 110                      |

---

## Objective
The goal is to **minimize total transportation costs** while ensuring:
1. Each distribution center receives the required number of packages.
2. No hub exceeds its package capacity.


In [1]:

# Import PuLP
from pulp import LpProblem, LpVariable, LpMinimize, lpSum, value



## Step 2: Define Input Data

We start by defining the key data for our problem, including transportation costs, hub capacities, and distribution center demands.


In [2]:

# Define costs as a dictionary
costs = {
    ('SFO', 'Seattle'): 80,
    ('SFO', 'Chicago'): 100,
    ('DEN', 'Chicago'): 70,
    ('DEN', 'Houston'): 90,
    ('AUS', 'Houston'): 50,
    ('AUS', 'Seattle'): 110
}

# Define hub capacities
capacities = {'SFO': 300, 'DEN': 250, 'AUS': 200}

# Define distribution center demands
demands = {'Seattle': 200, 'Chicago': 150, 'Houston': 250}



## Step 3: Define the Decision Variables

We create decision variables to represent the number of packages transported along each route.


In [3]:

# Generate decision variables for each route
routes = [(h, d) for h in capacities for d in demands if (h, d) in costs]
vars = LpVariable.dicts("Route", routes, lowBound=0, cat='Continuous')



## Step 4: Define the Objective Function

The objective is to minimize the total transportation costs across all routes.


In [4]:

# Create the Linear Programming problem
model = LpProblem("Minimize_Transportation_Costs", LpMinimize)

# Define the objective function
model += lpSum(vars[(h, d)] * costs[(h, d)] for (h, d) in routes), "Total_Cost"



## Step 5: Add Constraints

We add two types of constraints:
1. **Hub Capacity Constraints**: Ensure no hub ships more packages than its capacity.
2. **Distribution Center Demand Constraints**: Ensure each distribution center receives the exact number of required packages.


In [5]:

# Add hub capacity constraints
for h in capacities:
    model += lpSum(vars[(h, d)] for d in demands if (h, d) in costs) <= capacities[h], f"{h}_Capacity"

# Add distribution center demand constraints
for d in demands:
    model += lpSum(vars[(h, d)] for h in capacities if (h, d) in costs) == demands[d], f"{d}_Demand"


## Step 6: Solve the Model

We solve the optimization problem to find the optimal transportation plan.


In [6]:

# Solve the model
model.solve()


1


## Step 7: Display Results

Finally, we display the total transportation cost and the optimal number of packages to be shipped on each route.


In [7]:

# Print the optimal total cost
print("Optimal Total Cost:", value(model.objective))

# Print the optimal shipping quantities
print("Optimal Shipping Quantities:")
for (h, d) in routes:
    print(f"{h} to {d}: {vars[(h, d)].varValue}")


Optimal Total Cost: 41000.0
Optimal Shipping Quantities:
SFO to Seattle: 200.0
SFO to Chicago: 0.0
DEN to Chicago: 150.0
DEN to Houston: 50.0
AUS to Seattle: 0.0
AUS to Houston: 200.0
